In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
import os
import re

# ==========================================
# PRAWIDŁOWE NOTATKI DLA KLATEK (0 = Dobra, 1 = Zła)
# ==========================================
USER_NOTES_FRAMES = [
    (0, 519, 0),
    (519, 2062, 1),
    (2062, 2068, 0),
    (2068, 2142, 1),
    (2142, 5603, 0),  
    (5603, 6513, 1),
    (6513, 6515, 0),
    (6515, 7654, 1),
    (7654, 7679, 0),
    (7679, 7682, 1),
    (7682, 7909, 0)
]

# Funkcja przypisująca Twoją ocenę do konkretnej klatki
def get_ground_truth_for_frame(frame_num):
    for start, end, label in USER_NOTES_FRAMES:
        if start <= frame_num <= end:
            return label
    return 0  # Domyślnie dobra postawa, jeśli poza zakresem

# 1. Wczytanie danych z modelu (podmień na nazwę swojego pliku CSV z analizy klatek!)
CSV_PATH = "../../data/wyniki_z_klatek.csv"  # Skoków do góry z src/training/ do głównego data/

if not os.path.exists(CSV_PATH):
    print(f"BŁĄD: Nie znaleziono pliku {CSV_PATH}. Upewnij się, że nazwa jest poprawna.")
else:
    df = pd.read_csv(CSV_PATH)



In [4]:
 # 2. Wyciągnięcie numeru klatki z nazwy pliku (np. "frame_000123.jpg" -> 123)
    # Zakładam, że kolumna z nazwą pliku to "frame" (jeśli jest inna, zmień poniżej)
def extract_frame_num(filename):
    numbers = re.findall(r'\d+', str(filename))
    return int(numbers[0]) if numbers else 0

df['frame_num'] = df['frame'].apply(extract_frame_num)
    

In [5]:
 
    # 3. Dodanie Twoich notatek
df['ground_truth'] = df['frame_num'].apply(get_ground_truth_for_frame)

    # Czyszczenie z ewentualnych błędów detekcji (-1 to brak osoby w kadrze)
    # Pamiętaj, żeby upewnić się, że kolumna z oceną algorytmu nazywa się 'prediction' albo 'model_label'
col_name_model = 'prediction' if 'prediction' in df.columns else 'model_label'
df_clean = df[df[col_name_model] != -1].copy()

In [ ]:
  

    # 4. Obliczanie metryk
acc = accuracy_score(df_clean['ground_truth'], df_clean[col_name_model])
print(f"\n--- WYNIKI WALIDACJI (NA SUROWYCH KLATKACH) ---")
print(f"Przeanalizowano {len(df_clean)} klatek.")
print(f"Ogólna Dokładność (Accuracy): {acc*100:.2f}%")
print("\nSzczegółowy raport (Precision/Recall):")
print(classification_report(df_clean['ground_truth'], df_clean[col_name_model], target_names=['Dobra (0)', 'Zła (1)']))


In [ ]:

    # 5. Macierz Konfuzji
    cm = confusion_matrix(df_clean['ground_truth'], df_clean[col_name_model])
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Dobra', 'Zła'], yticklabels=['Dobra', 'Zła'])
    plt.title('Macierz Konfuzji (Metoda klatka po klatce)')
    plt.ylabel('Twoja ocena (Faktyczna)')
    plt.xlabel('Ocena Modelu AI (Przewidywana)')
    plt.show()
    

In [ ]:

    # 6. Opcjonalnie: Wykres w czasie
    plt.figure(figsize=(15, 4))
    plt.fill_between(df_clean['frame_num'], df_clean['ground_truth'], color='green', alpha=0.3, label='Twoja ocena')
    plt.step(df_clean['frame_num'], df_clean[col_name_model], where='post', color='red', alpha=0.7, label='Model AI')
    plt.title('Oś czasu predykcji względem ręcznej oceny (Numery klatek)')
    plt.xlabel('Numer klatki')
    plt.ylabel('Stan (0=Dobra, 1=Zła)')
    plt.legend()
    plt.show()